# Montar o app ExB "Mobilidade Urbana" (app 3) — v10
**Novidade da v10 — paradas oficiais da Mobilibus (mesmas do app Vou de Ônibus)**
- Busca na Mobilibus as 29 linhas e monta a camada **Paradas de Ônibus - Mobilidade Urbana** com as **paradas únicas**
  (a Pontos_bus_temp tinha 1.860 pontos = 1 cópia da parada por viagem). Cada parada guarda as linhas e os sentidos que passam nela.
- Publica/atualiza essa camada na pasta Mobilidade_Urbana (pública). **Rodar de novo = atualizar as paradas** a partir da Mobilibus.
- Web map: troca a Pontos_bus_temp pela nova camada (mesmo símbolo) e tira o mapa de calor.
- Interruptor **Mostrar paradas**: filtra pelo campo — mostra as paradas do **sentido exato** da linha selecionada
  (se o nome do sentido não bater, mostra as da linha inteira).

⚠ **O widget mudou → reinicie o client** → construtor **Ctrl+F5 → mudança pequena → Salvar → Publicar** → teste na **prévia** (https).
Saída: `Documents\\Saidas_Notebooks\\Montar_ExB_Mobilidade_v10.txt`

In [ ]:
# ===== CÉLULA 1 — Configuração =====
import os, json, datetime, shutil, copy
NB_NOME = "Montar_ExB_Mobilidade_v10"
EXB = r"C:\ExB_1.19"
EXB_APPS = os.path.join(EXB, "server", "public", "apps")
WIDGETS_DIR = os.path.join(EXB, "client", "your-extensions", "widgets")
APP_MODELO, APP_NOVO = "2", "3"
CFG_MODELO = os.path.join(EXB_APPS, APP_MODELO, "resources", "config", "config.json")
CFG_NOVO = os.path.join(EXB_APPS, APP_NOVO, "resources", "config", "config.json")
home = os.path.expanduser("~")
_c = [os.path.join(home, "Documents", "Saidas_Notebooks"), os.path.join(home, "OneDrive", "Documents", "Saidas_Notebooks")]
PASTA_SAIDA = next((c for c in _c if os.path.isdir(c)), _c[0]); os.makedirs(PASTA_SAIDA, exist_ok=True)
PASTA_BKP = os.path.join(PASTA_SAIDA, f"backup_exb_mobilidade_{datetime.datetime.now():%Y%m%d_%H%M%S}")
os.makedirs(PASTA_BKP, exist_ok=True)
ARQ_TXT = os.path.join(PASTA_SAIDA, NB_NOME + ".txt")
with open(ARQ_TXT, "w", encoding="utf-8") as f:
    f.write(f"{NB_NOME} — executado em {datetime.datetime.now():%d/%m/%Y %H:%M:%S}\n\n")
def log(*a):
    s = " ".join(str(x) for x in a); print(s)
    with open(ARQ_TXT, "a", encoding="utf-8") as f: f.write(s + "\n")
log("▶ CÉLULA 1 — Configuração: iniciando...")
for p in (CFG_MODELO, CFG_NOVO, WIDGETS_DIR):
    if not os.path.exists(p): raise RuntimeError("Não encontrado: " + p)
log("  Backups:", PASTA_BKP)
log("✔ CÉLULA 1 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 1B — Paradas da Mobilibus =====
log("\n▶ CÉLULA 1B — Busca das paradas na Mobilibus: iniciando...")
import requests, unicodedata, re, time
MOBI_URL = "https://editor.mobilibus.com/web/get-route-info"
MOBI_LINHAS = {"110":"fpt","115":"2cjn","120":"fqg","125":"fqa","130":"fqp","135":"fq7","140":"fq3","145":"fq4","150":"fpo",
 "155":"fr0","160":"fqt","165":"fqo","170":"fr1","175":"6ckh","195":"fpg","200":"fpm","205":"fps","210":"fqr","215":"hrte","220":"fpr",
 "230":"fq8","235":"fqk","240":"fph","270":"9t55","320":"fpu","400":"fqn","500":"fq6","998":"fqd","999":"fqb"}
def chave_sentido(s):
    s = unicodedata.normalize("NFKD", str(s or "")).encode("ascii", "ignore").decode().lower()
    s = s.replace("sentido para", "")
    return re.sub(r"[^a-z0-9]", "", s)
PARADAS = {}   # stopId -> dados
n_viagens = 0
for cod, rid in MOBI_LINHAS.items():
    r = requests.post(MOBI_URL, data=rid.encode(), headers={"Content-Type": "application/json;charset=UTF-8"}, timeout=60)
    r.raise_for_status()
    viagens = r.json()
    partes = []
    for v in viagens:
        paradas = v.get("stops") or []
        if not paradas: continue
        n_viagens += 1
        k = f"{cod}:{chave_sentido(v.get('tripName'))}"
        partes.append(f"{v.get('tripName')} ({len(paradas)})")
        for st in paradas:
            p = PARADAS.setdefault(st["stopId"], {"stop_id": st["stopId"], "nome": st.get("stopName") or "",
                                                 "lat": st["lat"], "lon": st["lon"], "linhas": set(), "viagens": set()})
            p["linhas"].add(cod); p["viagens"].add(k)
    log(f"  · {cod}: " + " | ".join(partes))
    time.sleep(0.3)
with open(os.path.join(PASTA_BKP, "paradas_mobilibus.json"), "w", encoding="utf-8") as f:
    json.dump({k: {**v, "linhas": sorted(v["linhas"]), "viagens": sorted(v["viagens"])} for k, v in PARADAS.items()},
              f, ensure_ascii=False, indent=1)
log(f"  Viagens: {n_viagens} | paradas únicas: {len(PARADAS)}")
log("✔ CÉLULA 1B concluída com sucesso.")


In [ ]:
# ===== CÉLULA 1C — Publicar/atualizar a camada de Paradas =====
log("\n▶ CÉLULA 1C — Camada Paradas de Ônibus no Portal: iniciando...")
import arcpy, zipfile
from arcgis.gis import GIS
from arcgis.features import FeatureLayer
gis = GIS("pro"); ME = gis.users.me
log("  Conectado como:", ME.username, "|", gis.url)
PASTA_PORTAL = "Mobilidade_Urbana"
PAR_TITULO = "Paradas de Ônibus - Mobilidade Urbana"
PAR_SERVICO = "Mobilidade_Paradas_Onibus"
def _nome_pasta(f):
    return f.get("title") if isinstance(f, dict) else (getattr(f, "name", None) or getattr(f, "title", None))
def pasta_obj():
    for f in (ME.folders or []):
        if _nome_pasta(f) == PASTA_PORTAL: return f
    return None
def itens_na_pasta():
    try: return list(ME.items(folder=PASTA_PORTAL, max_items=500))
    except Exception:
        f = pasta_obj(); return list(f.list()) if f is not None and hasattr(f, "list") else []
def adicionar(props, data=None):
    try:
        return gis.content.add(item_properties=props, data=data, folder=PASTA_PORTAL) if data else gis.content.add(item_properties=props, folder=PASTA_PORTAL)
    except Exception:
        from arcgis.gis import ItemProperties, ItemTypeEnum
        tipo = {"File Geodatabase": ItemTypeEnum.FILE_GEODATABASE, "Web Map": ItemTypeEnum.WEB_MAP}[props["type"]]
        ip = ItemProperties(title=props["title"], item_type=tipo, tags=props.get("tags"), snippet=props.get("snippet"), text=props.get("text"))
        job = pasta_obj().add(item_properties=ip, file=data) if data else pasta_obj().add(item_properties=ip)
        return job.result() if hasattr(job, "result") else job
def compartilhar(item):
    try: item.sharing.sharing_level = "EVERYONE"
    except Exception:
        try: item.share(everyone=True)
        except Exception as e: log("  ! não consegui compartilhar:", e)

registros = [{"stop_id": int(p["stop_id"]), "nome": p["nome"][:255],
              "linhas": "," + ",".join(sorted(p["linhas"], key=lambda x: int(x) if x.isdigit() else 0)) + ",",
              "viagens": ";" + ";".join(sorted(p["viagens"])) + ";", "n_viagens": len(p["viagens"]),
              "lon": p["lon"], "lat": p["lat"]} for p in PARADAS.values()]

existente = [i for i in itens_na_pasta() if i.type == "Feature Service" and i.title == PAR_TITULO]
if existente:
    fs = existente[0]
    lyr = fs.layers[0]
    log("  Camada já existe — atualizando os registros a partir da Mobilibus:", fs.id)
    lyr.delete_features(where="1=1")
    novos = [{"attributes": {k: r[k] for k in ("stop_id", "nome", "linhas", "viagens", "n_viagens")},
              "geometry": {"x": r["lon"], "y": r["lat"], "spatialReference": {"wkid": 4326}}} for r in registros]
    for i in range(0, len(novos), 250):
        lyr.edit_features(adds=novos[i:i+250])
else:
    trab = os.path.join(PASTA_BKP, "paradas"); os.makedirs(trab, exist_ok=True)
    gdb = os.path.join(trab, PAR_SERVICO + ".gdb")
    arcpy.management.CreateFileGDB(trab, PAR_SERVICO + ".gdb")
    fc = os.path.join(gdb, PAR_SERVICO)
    arcpy.management.CreateFeatureclass(gdb, PAR_SERVICO, "POINT", spatial_reference=arcpy.SpatialReference(4326))
    for nome, tipo, tam in (("stop_id", "LONG", None), ("nome", "TEXT", 255), ("linhas", "TEXT", 1000),
                            ("viagens", "TEXT", 4000), ("n_viagens", "SHORT", None)):
        arcpy.management.AddField(fc, nome, tipo, field_length=tam)
    with arcpy.da.InsertCursor(fc, ["SHAPE@XY", "stop_id", "nome", "linhas", "viagens", "n_viagens"]) as cur:
        for r in registros:
            cur.insertRow(((r["lon"], r["lat"]), r["stop_id"], r["nome"], r["linhas"], r["viagens"], r["n_viagens"]))
    arcpy.management.ClearWorkspaceCache()
    z = os.path.join(trab, PAR_SERVICO + ".zip")
    with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
        for raiz, _, arqs in os.walk(gdb):
            for a in arqs:
                if a.lower().endswith(".lock"): continue
                full = os.path.join(raiz, a)
                zf.write(full, os.path.join(os.path.basename(gdb), os.path.relpath(full, gdb)))
    props = {"title": PAR_TITULO, "type": "File Geodatabase",
             "tags": "mobilidade, ônibus, paradas, pontos de ônibus, Mobilibus, Apucarana, GeoAPUC",
             "snippet": "Paradas de ônibus de Apucarana (fonte: Mobilibus / app Vou de Ônibus), com as linhas e sentidos que passam em cada uma."}
    gdb_item = adicionar(props, z)
    fs = gdb_item.publish(publish_parameters={"name": PAR_SERVICO}, file_type="fileGeodatabase")
    try:
        fs.update(item_properties={"title": PAR_TITULO, "tags": props["tags"], "snippet": props["snippet"]})
        if fs.ownerFolder is None: fs.move(PASTA_PORTAL)
    except Exception as e: log("  ! ajuste de propriedades:", e)
    log("  Publicada:", fs.id)
compartilhar(fs)
PARADAS_ID = fs.id
PARADAS_URL = fs.url.rstrip("/") + "/0"
n = fs.layers[0].query(where="1=1", return_count_only=True)
log(f"  Paradas no Portal: {n} (esperado {len(registros)}) → {'OK' if n == len(registros) else 'CONFERIR'}")
log("  URL:", PARADAS_URL)
log("✔ CÉLULA 1C concluída com sucesso.")


In [ ]:
# ===== CÉLULA 1D — Web map: Paradas no lugar da Pontos_bus_temp, sem mapa de calor =====
log("\n▶ CÉLULA 1D — Atualização do web map: iniciando...")
WEBMAP_ID = "c80cb05e18a645609f8f84740078b859"
wm_item = gis.content.get(WEBMAP_ID)
wm = wm_item.get_data()
with open(os.path.join(PASTA_BKP, "webmap_antes_paradas.json"), "w", encoding="utf-8") as f:
    json.dump(wm, f, ensure_ascii=False, indent=2)
ops = wm.get("operationalLayers", [])
antigo = next((ol for ol in ops if ol.get("id") in ("pontos_onibus", "paradas_onibus")), None)
desenho = (antigo or {}).get("layerDefinition", {}).get("drawingInfo")
nova = {"id": "paradas_onibus", "title": "Paradas de ônibus", "url": PARADAS_URL, "itemId": PARADAS_ID,
        "layerType": "ArcGISFeatureLayer", "visibility": False, "opacity": 1, "disablePopup": True,
        "popupInfo": {"title": "{nome}", "fieldInfos": [{"fieldName": "nome", "label": "Parada", "visible": True}]}}
if desenho: nova["layerDefinition"] = {"drawingInfo": desenho}
novos_ops = []
for ol in ops:
    if ol.get("id") == "calor_pontos":
        log("  · removida: mapa de calor"); continue
    if ol is antigo:
        novos_ops.append(nova); log(f"  · '{ol.get('title')}' trocada por 'Paradas de ônibus'"); continue
    novos_ops.append(ol)
if antigo is None:
    novos_ops.append(nova); log("  · 'Paradas de ônibus' adicionada")
wm["operationalLayers"] = novos_ops
wm_item.update(item_properties={"text": json.dumps(wm)})
log("  Camadas agora:", [o.get("id") for o in novos_ops])
log("✔ CÉLULA 1D concluída com sucesso.")


In [ ]:
# ===== CÉLULA 2 — Widget próprio 'Horários da Linha' =====
log("\n▶ CÉLULA 2 — Criação do widget Horários da Linha: iniciando...")
ARQUIVOS_WIDGET = {
    'manifest.json': r'''{
  "name": "horarios-linha",
  "label": "Horários da Linha",
  "type": "widget",
  "version": "1.19.0",
  "exbVersion": "1.19.0",
  "author": "GeoAPUC",
  "description": "Mostra a tabela de horários (dias úteis) da linha de ônibus selecionada na lista, com a próxima saída em destaque.",
  "copyright": "",
  "license": "",
  "properties": {},
  "translatedLocales": [
    "en"
  ],
  "defaultSize": {
    "width": 380,
    "height": 560
  },
  "dependency": [
    "jimu-arcgis"
  ]
}''',
    'config.json': r'''{
  "dataSourceId": "dataSource_1-linhas_onibus",
  "linhasUrl": "https://geo.apucarana.pr.gov.br/server/rest/services/Hosted/Mobilidade_Linhas_Onibus/FeatureServer/0",
  "horariosUrl": "https://geo.apucarana.pr.gov.br/server/rest/services/Hosted/Mobilidade_Horarios_Onibus/FeatureServer/0",
  "tema": "azul",
  "mobilibusUrl": "https://editor.mobilibus.com/web/detalhes-linha/4abp9",
  "mobilibusLinhas": {
    "110": "fpt",
    "115": "2cjn",
    "120": "fqg",
    "125": "fqa",
    "130": "fqp",
    "135": "fq7",
    "140": "fq3",
    "145": "fq4",
    "150": "fpo",
    "155": "fr0",
    "160": "fqt",
    "165": "fqo",
    "170": "fr1",
    "175": "6ckh",
    "195": "fpg",
    "200": "fpm",
    "205": "fps",
    "210": "fqr",
    "215": "hrte",
    "220": "fpr",
    "230": "fq8",
    "235": "fqk",
    "240": "fph",
    "270": "9t55",
    "320": "fpu",
    "400": "fqn",
    "500": "fq6",
    "998": "fqd",
    "999": "fqb"
  },
  "paradasUrl": "https://geo.apucarana.pr.gov.br/server/rest/services/Hosted/Mobilidade_Paradas_Onibus/FeatureServer/0",
  "paradasLayerId": "paradas_onibus",
  "paradasDistanciaM": 30,
  "paradasLigadoAoAbrir": false,
  "paradasModo": "campo"
}''',
    'icon.svg': r'''<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 16 16" width="16" height="16"><rect x="3" y="1.5" width="10" height="11" rx="2" fill="none" stroke="#000" stroke-width="1.2"/><rect x="4.6" y="3.2" width="6.8" height="3.6" rx=".6" fill="#000"/><circle cx="5.5" cy="9.6" r=".9" fill="#000"/><circle cx="10.5" cy="9.6" r=".9" fill="#000"/><rect x="4" y="12.5" width="2" height="2" rx=".5" fill="#000"/><rect x="10" y="12.5" width="2" height="2" rx=".5" fill="#000"/></svg>
''',
    'src/runtime/widget.tsx': r'''// Horários da Linha – GeoAPUC (v6: paradas filtradas pelo campo 'viagens' da camada de Paradas (Mobilibus))
// Mostra os horários (dias úteis) da linha selecionada na lista de linhas.
// Escrito sem JSX (React.createElement) para não depender da configuração de compilação.
import { React, getAppStore, type AllWidgetProps } from 'jimu-core'
import { JimuMapViewComponent } from 'jimu-arcgis'

const h = React.createElement
const { useState, useEffect } = React

const TEMAS: any = {
  escuro: { fundo: '#0f1e33', card: '#16294a', borda: 'rgba(255,255,255,0.10)', texto: '#ffffff', suave: '#9fb0cc',
            destaque: '#F5A623', azul: '#4285F4', verde: '#1D9E75', chip: '#1f3b66', aviso: 'rgba(245,166,35,0.14)' },
  azul: { fundo: '#4463a9', card: 'rgba(255,255,255,0.13)', borda: 'rgba(255,255,255,0.25)', texto: '#ffffff', suave: '#dbe4f5',
          destaque: '#F5A623', azul: '#1b2a4a', verde: '#1D9E75', chip: 'rgba(255,255,255,0.20)', aviso: 'rgba(245,166,35,0.22)' },
  claro: { fundo: '#ffffff', card: '#f4f6fb', borda: '#dfe5ef', texto: '#1b2a4a', suave: '#6b7690',
           destaque: '#4463a9', azul: '#4463a9', verde: '#1f9d55', chip: '#e6ecf8', aviso: '#FFF4E0' }
}

function hhmm (v: any): string {
  const s = String(v ?? '').trim()
  const m = s.match(/^(\d{1,2}):(\d{2})/)
  return m ? `${m[1].padStart(2, '0')}:${m[2]}` : s
}
function minutos (v: string): number {
  const m = v.match(/^(\d{2}):(\d{2})/)
  return m ? Number(m[1]) * 60 + Number(m[2]) : -1
}
function chaveSentido (v: any): string {
  return String(v ?? '').normalize('NFD').replace(/[\u0300-\u036f]/g, '').toLowerCase()
    .replace(/sentido para/g, '').replace(/[^a-z0-9]/g, '')
}
async function contar (url: string, where: string): Promise<number> {
  const q = `${url}/query?where=${encodeURIComponent(where)}&returnCountOnly=true&f=json`
  const j: any = await fetch(q).then(r => r.json())
  if (j.error) throw new Error(j.error.message || 'erro na contagem')
  return Number(j.count || 0)
}
function aspas (v: any): string { return String(v ?? '').replace(/'/g, "''") }

async function consultar (url: string, where: string, campos: string): Promise<any[]> {
  const q = `${url}/query?where=${encodeURIComponent(where)}&outFields=${campos}&returnGeometry=false&f=json`
  const j: any = await fetch(q).then(r => r.json())
  if (j.error) throw new Error(j.error.message || 'erro na consulta')
  return (j.features || []).map((f: any) => f.attributes || {})
}

async function geometriaLinha (urlLin: string, oid: string): Promise<any> {
  const q = `${urlLin}/query?where=${encodeURIComponent('objectid=' + Number(oid))}&returnGeometry=true&outFields=objectid&f=json`
  const j: any = await fetch(q).then(r => r.json())
  if (j.error) throw new Error(j.error.message || 'erro na geometria')
  const f = (j.features || [])[0]
  if (!f?.geometry) throw new Error('linha sem geometria')
  const g = f.geometry
  if (!g.spatialReference) g.spatialReference = j.spatialReference
  return g
}
async function paradasPerto (urlPar: string, geom: any, metros: number): Promise<number[]> {
  const corpo = new URLSearchParams({
    geometry: JSON.stringify(geom), geometryType: 'esriGeometryPolyline', spatialRel: 'esriSpatialRelIntersects',
    distance: String(metros), units: 'esriSRUnit_Meter', returnIdsOnly: 'true', f: 'json'
  })
  const j: any = await fetch(`${urlPar}/query`, { method: 'POST', body: corpo }).then(r => r.json())
  if (j.error) throw new Error(j.error.message || 'erro nas paradas')
  return j.objectIds || []
}

function relogio (cor: string, tam: number) {
  return h('svg', { width: tam, height: tam, viewBox: '0 0 24 24', style: { flex: 'none' } },
    h('circle', { cx: 12, cy: 12, r: 10, fill: '#ffffff', stroke: cor, strokeWidth: 2 }),
    h('path', { d: 'M12 6v6l4 2', fill: 'none', stroke: '#c0392b', strokeWidth: 2, strokeLinecap: 'round' }))
}

export default function Widget (props: AllWidgetProps<any>) {
  const cfg: any = props.config || {}
  const dsId: string = cfg.dataSourceId || 'dataSource_1-linhas_onibus'
  const urlLin: string = cfg.linhasUrl
  const urlHor: string = cfg.horariosUrl
  const T: any = TEMAS[cfg.tema] || TEMAS.escuro
  const mbBase: string = cfg.mobilibusUrl || 'https://editor.mobilibus.com/web/detalhes-linha/4abp9'
  const mbCod: any = cfg.mobilibusLinhas || {}
  const linkTempoReal = (cod: any): string => {
    const c = mbCod[String(cod ?? '').trim()]
    return c ? `${mbBase}#${c}` : mbBase
  }
  const botaoTempoReal = (cod: any, texto: string) => h('a', {
    href: linkTempoReal(cod), target: '_blank', rel: 'noopener noreferrer',
    style: { display: 'flex', alignItems: 'center', justifyContent: 'center', gap: 8, padding: '9px 12px', borderRadius: 999,
             background: '#d64545', color: '#ffffff', fontWeight: 700, fontSize: 13, textDecoration: 'none' }
  }, h('span', { style: { width: 9, height: 9, borderRadius: '50%', background: '#ffffff', display: 'inline-block' } }), texto)

  // id do registro selecionado na lista (lido do estado do app e acompanhado por assinatura)
  const lerSelecao = (): string | null => {
    try {
      const inf: any = (getAppStore().getState() as any)?.dataSourcesInfo || {}
      for (const k of Object.keys(inf)) {
        if (k === dsId || k.startsWith(dsId + '-')) {
          const ids = inf[k]?.selectedIds
          if (ids && ids.length) return String(ids[0])
        }
      }
    } catch (e) { console.warn('[Horários da Linha] seleção', e) }
    return null
  }
  const [oid, setOid] = useState<string | null>(lerSelecao())
  useEffect(() => {
    const store: any = getAppStore()
    const desfazer = store.subscribe(() => {
      const novo = lerSelecao()
      setOid(atual => (atual === novo ? atual : novo))
    })
    return () => { try { desfazer() } catch (e) {} }
  }, [dsId])

  const [linha, setLinha] = useState<any>(null)
  const [hor, setHor] = useState<any[]>([])
  const [planoB, setPlanoB] = useState<boolean>(false)
  const [carregando, setCarregando] = useState<boolean>(false)
  const [erro, setErro] = useState<string | null>(null)
  const [agora, setAgora] = useState<Date>(new Date())

  useEffect(() => { const t = setInterval(() => setAgora(new Date()), 60000); return () => clearInterval(t) }, [])

  // ---- interruptor "Mostrar paradas" ----
  const urlPar: string = cfg.paradasUrl
  const idCamadaParadas: string = cfg.paradasLayerId || 'pontos_onibus'
  const metros: number = Number(cfg.paradasDistanciaM || 30)
  const [jmv, setJmv] = useState<any>(null)
  const [mostrar, setMostrar] = useState<boolean>(!!cfg.paradasLigadoAoAbrir)
  const [nParadas, setNParadas] = useState<number | null>(null)
  useEffect(() => {
    const v: any = jmv?.view
    if (!v || !urlPar) return
    const lyr: any = v.map.allLayers.find((l: any) => l.id === idCamadaParadas)
    if (!lyr) { console.warn('[Horários da Linha] camada de paradas não encontrada:', idCamadaParadas); return }
    let vivo = true
    if (!mostrar) { lyr.visible = false; setNParadas(null); return }
    ;(async () => {
      try {
        await lyr.load()
        const oidCampo = lyr.objectIdField || 'objectid'
        if (!oid) { lyr.definitionExpression = '1=0'; lyr.visible = true; if (vivo) setNParadas(0); return }
        if (cfg.paradasModo === 'campo') {
          // camada de Paradas (Mobilibus): campos 'viagens' (;cod:sentido;) e 'linhas' (,cod,)
          const l = (await consultar(urlLin, `objectid=${Number(oid)}`, 'cod_linha,sentido'))[0] || {}
          const cod = String(l.cod_linha ?? '').trim()
          let where = `viagens LIKE '%;${aspas(cod + ':' + chaveSentido(l.sentido))};%'`
          let n = await contar(urlPar, where)
          if (!n) { where = `linhas LIKE '%,${aspas(cod)},%'`; n = await contar(urlPar, where) }
          if (!vivo) return
          lyr.definitionExpression = n ? where : '1=0'
          lyr.visible = true
          setNParadas(n)
        } else {
          const geom = await geometriaLinha(urlLin, oid)
          const ids = await paradasPerto(urlPar, geom, metros)
          if (!vivo) return
          lyr.definitionExpression = ids.length ? `${oidCampo} IN (${ids.join(',')})` : '1=0'
          lyr.visible = true
          setNParadas(ids.length)
        }
      } catch (e) {
        console.warn('[Horários da Linha] paradas', e)
        if (vivo) { setNParadas(null) }
      }
    })()
    return () => { vivo = false }
  }, [jmv, mostrar, oid, urlPar, urlLin, idCamadaParadas, metros])

  const interruptor = h('div', {
    onClick: () => setMostrar(!mostrar), role: 'switch', 'aria-checked': mostrar, title: 'Mostrar no mapa só as paradas da linha selecionada',
    style: { display: 'flex', alignItems: 'center', gap: 8, cursor: 'pointer', userSelect: 'none', fontSize: 12, color: T.texto }
  },
    h('span', null, 'Mostrar paradas' + (mostrar && nParadas !== null ? ` (${nParadas})` : '')),
    h('span', { style: { width: 38, height: 20, borderRadius: 999, position: 'relative', flex: 'none',
                         background: mostrar ? T.destaque : 'rgba(255,255,255,0.35)', transition: 'background .2s' } },
      h('span', { style: { position: 'absolute', top: 2, left: mostrar ? 20 : 2, width: 16, height: 16, borderRadius: '50%',
                           background: '#ffffff', boxShadow: '0 1px 2px rgba(0,0,0,.3)', transition: 'left .2s' } })))
  const mapaId: any = (props as any).useMapWidgetIds?.[0]
  const conector = mapaId ? h(JimuMapViewComponent as any, { useMapWidgetId: mapaId, onActiveViewChange: (v: any) => setJmv(v) }) : null

  useEffect(() => {
    let vivo = true
    if (!oid) { setLinha(null); setHor([]); return }
    setCarregando(true); setErro(null)
    ;(async () => {
      try {
        const l = (await consultar(urlLin, `objectid=${Number(oid)}`, 'objectid,id_rota,cod_linha,nome,sentido'))[0]
        if (!l) throw new Error('linha não encontrada')
        let rows = await consultar(urlHor, `id_rota='${aspas(l.id_rota)}'`, 'id_rota,ponto,horario,sentido,observacao')
        let b = false
        if (!rows.length) {
          const cod = String(l.id_rota || '').split('_')[0]
          rows = (await consultar(urlHor, `id_rota LIKE '${aspas(cod)}%'`, 'id_rota,ponto,horario,sentido,observacao'))
            .filter((r: any) => String(r.id_rota || '').startsWith(cod + '_'))
          b = rows.length > 0
        }
        rows.forEach((r: any) => { r._h = hhmm(r.horario); r._m = minutos(r._h) })
        rows.sort((a: any, z: any) => a._m - z._m)
        if (vivo) { setLinha(l); setHor(rows); setPlanoB(b); setCarregando(false) }
      } catch (e: any) {
        console.warn('[Horários da Linha]', e)
        if (vivo) { setErro('Não foi possível carregar os horários.'); setCarregando(false) }
      }
    })()
    return () => { vivo = false }
  }, [oid, urlLin, urlHor])

  const chip = (txt: string, bg: string, cor = '#ffffff') =>
    h('span', { style: { display: 'inline-block', padding: '2px 10px', borderRadius: 999, background: bg, color: cor, fontSize: 11, fontWeight: 700 } }, txt)
  const rotulo = (txt: string) => h('div', { style: { fontSize: 10, letterSpacing: 0.6, color: T.suave, textTransform: 'uppercase' } }, txt)

  const caixa = (filhos: any[]) => h('div', {
    className: 'jimu-widget horarios-linha',
    style: { width: '100%', height: '100%', boxSizing: 'border-box', background: T.fundo, color: T.texto, borderRadius: 12,
             padding: 14, overflowY: 'auto', fontFamily: 'inherit', display: 'flex', flexDirection: 'column', gap: 12,
             border: `1px solid ${T.borda}` }
  }, conector,
    h('div', { style: { display: 'flex', alignItems: 'center', justifyContent: 'space-between', gap: 8 } },
      h('div', { style: { fontSize: 15, fontWeight: 600, color: T.texto } }, 'Tabela de Horários'), interruptor),
    ...filhos)

  if (!oid) {
    return caixa([h('div', { style: { margin: 'auto', textAlign: 'center', color: T.suave, fontSize: 13, lineHeight: 1.5 } },
      h('div', { style: { fontSize: 34, marginBottom: 6 } }, '🚌'), 'Selecione uma linha na lista', h('br'), 'para ver os horários.'),
      botaoTempoReal(null, 'Onde está meu ônibus? · tempo real')])
  }
  if (carregando) return caixa([h('div', { style: { margin: 'auto', color: T.suave } }, 'Carregando horários…')])
  if (erro || !linha) return caixa([h('div', { style: { margin: 'auto', color: '#e57373' } }, erro || '—')])

  try {
    // próxima saída (só em dias úteis)
    const dow = agora.getDay()
    const diaUtil = dow >= 1 && dow <= 5
    const minAgora = agora.getHours() * 60 + agora.getMinutes()
    const proxima = diaUtil ? hor.find((r: any) => r._m >= minAgora) : null
    let destaque: string = proxima ? proxima._h : (hor[0]?._h || '—')
    let sub = ''
    if (!hor.length) sub = 'Sem horários na base da Prefeitura — veja no botão de tempo real.'
    else if (!diaUtil) sub = 'Fim de semana — horários válidos para dias úteis. Primeira saída:'
    else if (proxima) { const d = proxima._m - minAgora; sub = d === 0 ? 'Saindo agora' : `Próxima saída · em ${d >= 60 ? Math.floor(d / 60) + ' h ' : ''}${d % 60} min` }
    else sub = 'Não há mais saídas hoje. Primeira saída amanhã:'

    // agrupa por sentido (plano B) e por ponto
    const grupos: any = {}
    hor.forEach((r: any) => {
      const k = (planoB ? (r.sentido || 'Outro sentido') + ' · ' : '') + (r.ponto || '—')
      ;(grupos[k] = grupos[k] || []).push(r)
    })
    const obs = Array.from(new Set(hor.map((r: any) => String(r.observacao || '').trim()).filter((x: string) => x && x !== '-')))

    return caixa([
      // cabeçalho da linha
      h('div', { style: { display: 'flex', alignItems: 'center', gap: 12, background: T.card, borderRadius: 10, padding: 10 } },
        h('div', { style: { minWidth: 46, textAlign: 'center', padding: '8px 6px', borderRadius: 8, border: `1px solid ${T.destaque}`,
                            color: T.destaque, fontWeight: 800, fontSize: 17, background: 'rgba(245,166,35,0.10)' } }, String(linha.cod_linha ?? '')),
        h('div', { style: { display: 'flex', flexDirection: 'column', gap: 4, minWidth: 0 } },
          h('div', { style: { fontSize: 16, fontWeight: 700, lineHeight: 1.2 } }, linha.nome || ''),
          h('div', { style: { display: 'flex', gap: 6, flexWrap: 'wrap' } }, chip('Dias úteis', T.azul), chip(linha.sentido || '', T.verde)))),
      botaoTempoReal(linha.cod_linha, `Onde está o ônibus da linha ${linha.cod_linha}? · tempo real`),
      // próxima saída
      h('div', { style: { display: 'flex', alignItems: 'center', gap: 12, background: T.card, borderRadius: 10, padding: '10px 12px' } },
        relogio(T.suave, 34),
        h('div', null, rotulo(sub || 'Horário'), h('div', { style: { fontSize: 32, fontWeight: 800, lineHeight: 1.1 } }, destaque))),
      planoB
        ? h('div', { style: { background: T.aviso, borderLeft: `4px solid ${T.destaque}`, borderRadius: 6, padding: '6px 10px', fontSize: 12, color: T.texto } },
            `Este sentido não tem horários próprios. Abaixo, os horários de saída da linha ${linha.cod_linha} nos outros sentidos.`)
        : null,
      // todos os horários
      ...Object.keys(grupos).map((k: string) => h('div', { key: k, style: { display: 'flex', flexDirection: 'column', gap: 6 } },
        rotulo('Ponto · ' + k),
        h('div', { style: { display: 'flex', flexWrap: 'wrap', gap: 6 } },
          ...grupos[k].map((r: any, i: number) => {
            const passou = diaUtil && r._m < minAgora
            const ehProx = proxima && r === proxima
            return h('span', { key: i, style: { padding: '4px 9px', borderRadius: 6, fontWeight: 700, fontSize: 13,
              background: ehProx ? T.destaque : T.chip, color: ehProx ? '#1b2a4a' : T.texto, opacity: passou ? 0.4 : 1 } }, r._h)
          })))),
      obs.length ? h('div', null, rotulo('Observação'), h('div', { style: { fontSize: 12, color: T.suave } }, obs.join(' · '))) : null,
      h('div', { style: { fontSize: 11, color: T.suave, marginTop: 'auto' } }, `${hor.length} horários · fonte: setor de transportes`)
    ])
  } catch (e: any) {
    console.error('[Horários da Linha]', e)
    return caixa([h('div', { style: { color: '#e57373', fontSize: 12 } }, 'Erro no widget: ' + String(e?.message || e))])
  }
}
''',
    'src/runtime/translations/default.ts': r'''export default {
  _widgetLabel: 'Horários da Linha'
}
''',
}

destino = os.path.join(WIDGETS_DIR, "horarios-linha")
if os.path.isdir(destino):
    shutil.copytree(destino, os.path.join(PASTA_BKP, "widget_horarios-linha_antes"), dirs_exist_ok=True)
    log("  Widget já existia — backup feito e arquivos atualizados.")
for rel, conteudo in ARQUIVOS_WIDGET.items():
    p = os.path.join(destino, *rel.split("/"))
    os.makedirs(os.path.dirname(p), exist_ok=True)
    with open(p, "w", encoding="utf-8", newline="\n") as f:
        f.write(conteudo)
    log("  ·", p)
log("✔ CÉLULA 2 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 3 — Montar o config (casca do app 2 + conteúdo de Mobilidade) =====
log("\n▶ CÉLULA 3 — Montagem do config: iniciando...")
# ---- função que copia a "casca" homologada (cabeçalho do app CCTV) para o app novo ----
import json, copy
def montar_com_casca(cctv, body, titulo="Mobilidade Urbana", mapa_novo="widget_mapa", ds_novo="dataSource_1",
                     mapa_antigo="widget_2", ds_antigo="dataSource_4", excluir=("widgets/placar-cameras/",)):
    W, L = cctv["widgets"], cctv["layouts"]
    # 1) percorre a árvore do cabeçalho
    wids, lids = [], []
    def walk(lid):
        lids.append(lid)
        for it in L[lid]["content"].values():
            wid = it.get("widgetId")
            if not wid or wid in wids: continue
            wids.append(wid)
            for lv in (W[wid].get("layouts") or {}).values():
                for l2 in set(lv.values()):
                    if l2 in L and l2 not in lids: walk(l2)
    hdr_lay = cctv["header"]["layout"]["LARGE"]
    walk(hdr_lay)
    excl = [w for w in wids if W[w]["uri"] in excluir]
    wids = [w for w in wids if w not in excl]
    sub_w = {w: copy.deepcopy(W[w]) for w in wids}
    sub_l = {l: copy.deepcopy(L[l]) for l in lids}
    for l in sub_l.values():   # tira widgets excluídos dos layouts (ex.: placar das câmeras)
        for k in [k for k, it in l["content"].items() if it.get("widgetId") in excl]:
            del l["content"][k]
            if k in l.get("order", []): l["order"].remove(k)
    for w in sub_w.values():
        b = ((w.get("config") or {}).get("behavior") or {})
        for e in excl: (b.get("size") or {}).pop(e, None)
        if w["uri"] == "widgets/arcgis/map-layers/":
            w["config"] = {"useMapWidget": True}
        if w["uri"] == "widgets/common/search/":
            # mantém só a busca de endereço (geocodificador); tira fontes ligadas às camadas do mapa do CCTV
            removidos = []
            for k, v in (w["config"].get("dataSourceConfigWithMapCentric") or {}).items():
                fica = [d for d in v.get("dataSourceConfig", []) if not d.get("useDataSource")]
                removidos += [d.get("outputDataSourceId") for d in v.get("dataSourceConfig", []) if d.get("useDataSource")]
                v["dataSourceConfig"] = fica
            for k in ("dataSourceConfig",):
                if isinstance(w["config"].get(k), list):
                    removidos += [d.get("outputDataSourceId") for d in w["config"][k] if d.get("useDataSource")]
                    w["config"][k] = [d for d in w["config"][k] if not d.get("useDataSource")]
            w["useDataSources"] = [u for u in (w.get("useDataSources") or []) if not str(u.get("dataSourceId", "")).startswith(ds_antigo + "-")]
            if not w["useDataSources"]: w.pop("useDataSources", None)
            w["outputDataSources"] = [o for o in (w.get("outputDataSources") or []) if o not in removidos]
    # 2) saídas de dados da busca + mensagens da busca
    saidas = set()
    for w in sub_w.values():
        saidas.update(w.get("outputDataSources") or [])
    out_ds = {k: copy.deepcopy(v) for k, v in cctv.get("dataSources", {}).items() if k in saidas}
    msgs = {k: copy.deepcopy(v) for k, v in cctv.get("messageConfigs", {}).items() if v.get("widgetId") in wids}
    # 3) remapeia mapa/fonte do CCTV para os do app novo
    pacote = json.dumps({"w": sub_w, "l": sub_l, "d": out_ds, "m": msgs}, ensure_ascii=False)
    pacote = pacote.replace(f"{mapa_antigo}-{ds_antigo}", f"{mapa_novo}-{ds_novo}").replace(f'"{mapa_antigo}"', f'"{mapa_novo}"')
    pacote = pacote.replace(f'"{ds_antigo}"', f'"{ds_novo}"')
    p = json.loads(pacote)
    # 4) título
    for w in p["w"].values():
        if w["uri"] == "widgets/common/text/" and "Sistema de Monitoramento" in w["config"].get("text", ""):
            w["config"]["text"] = w["config"]["text"].replace("Sistema de Monitoramento", titulo)
    # 5) monta o config final
    cfg = {k: copy.deepcopy(cctv[k]) for k in ("exbVersion", "mainSizeMode", "theme", "customTheme", "forBuilderAttributes",
                                               "template", "attributes", "sharedThemeVariables", "utilities", "mainLocale")
           if k in cctv}
    cfg["header"] = copy.deepcopy(cctv["header"])
    cfg.update({"pages": body["pages"], "pageStructure": body["pageStructure"],
                "layouts": {**p["l"], **body["layouts"]}, "widgets": {**p["w"], **body["widgets"]},
                "widgetsManifest": {}, "views": {}, "sections": {}, "dialogs": {}, "controllerPanels": {},
                "messageConfigs": {**p["m"], **body["messageConfigs"]},
                "dataSources": {**p["d"], **body["dataSources"]}})
    # conferências
    for lay in cfg["layouts"].values():
        for it in lay["content"].values():
            assert it["widgetId"] in cfg["widgets"], it["widgetId"]
    s = json.dumps(cfg)
    assert f'"{mapa_antigo}"' not in s and f"{ds_antigo}" not in s, "sobrou referência ao app de CCTV"
    return cfg, [W[w]["label"] for w in wids], [W[w]["label"] for w in excl]

BODY = json.loads(r'''{
  "pages": {
    "page_0": {
      "id": "page_0",
      "mode": "FIT_WINDOW",
      "type": "NORMAL",
      "layout": {
        "LARGE": "layout_p0"
      },
      "isDefault": true,
      "isVisible": true,
      "label": "Linhas e Horários",
      "header": true
    }
  },
  "pageStructure": [
    {
      "page_0": []
    }
  ],
  "layouts": {
    "layout_card": {
      "type": "FIXED",
      "content": {
        "0": {
          "type": "WIDGET",
          "widgetId": "widget_card_icone",
          "isPending": false,
          "bbox": {
            "left": "2%",
            "right": "",
            "top": "12px",
            "bottom": "",
            "width": "40px",
            "height": "40px"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false,
            "lockParent": true
          }
        },
        "1": {
          "type": "WIDGET",
          "widgetId": "widget_card_texto",
          "isPending": false,
          "bbox": {
            "left": "14%",
            "right": "2.0%",
            "top": "2px",
            "bottom": "",
            "width": "84%",
            "height": "60px"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false,
            "lockParent": true
          }
        }
      },
      "order": [
        "0",
        "1"
      ],
      "label": "Padrão"
    },
    "layout_mapa1": {
      "type": "FIXED",
      "content": {},
      "order": []
    },
    "layout_p0": {
      "type": "FIXED",
      "content": {
        "0": {
          "type": "WIDGET",
          "widgetId": "widget_tit_lista",
          "isPending": false,
          "bbox": {
            "left": "1%",
            "right": "75.0%",
            "top": "1%",
            "bottom": "91.0%",
            "width": "24%",
            "height": "8%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "1": {
          "type": "WIDGET",
          "widgetId": "widget_lista",
          "isPending": false,
          "bbox": {
            "left": "1%",
            "right": "75.0%",
            "top": "10%",
            "bottom": "1.0%",
            "width": "24%",
            "height": "89%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "2": {
          "type": "WIDGET",
          "widgetId": "widget_mapa",
          "isPending": false,
          "bbox": {
            "left": "25.8%",
            "right": "26.8%",
            "top": "1%",
            "bottom": "1.0%",
            "width": "47.4%",
            "height": "98%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "3": {
          "type": "WIDGET",
          "widgetId": "widget_horarios",
          "isPending": false,
          "bbox": {
            "left": "74%",
            "right": "1.0%",
            "top": "1%",
            "bottom": "43.0%",
            "width": "25%",
            "height": "56%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "4": {
          "type": "WIDGET",
          "widgetId": "widget_tit_serv",
          "isPending": false,
          "bbox": {
            "left": "74%",
            "right": "1.0%",
            "top": "58%",
            "bottom": "38.0%",
            "width": "25%",
            "height": "4%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "5": {
          "type": "WIDGET",
          "widgetId": "widget_card_integra",
          "isPending": false,
          "bbox": {
            "left": "74%",
            "right": "1.0%",
            "top": "62.5%",
            "bottom": "26.5%",
            "width": "25%",
            "height": "11%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "6": {
          "type": "WIDGET",
          "widgetId": "widget_card_app",
          "isPending": false,
          "bbox": {
            "left": "74%",
            "right": "1.0%",
            "top": "74.5%",
            "bottom": "14.0%",
            "width": "25%",
            "height": "11.5%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "7": {
          "type": "WIDGET",
          "widgetId": "widget_btn_gplay",
          "isPending": false,
          "bbox": {
            "left": "75%",
            "right": "13.7%",
            "top": "81.2%",
            "bottom": "15.0%",
            "width": "11.3%",
            "height": "3.8%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "8": {
          "type": "WIDGET",
          "widgetId": "widget_btn_appstore",
          "isPending": false,
          "bbox": {
            "left": "87%",
            "right": "1.7%",
            "top": "81.2%",
            "bottom": "15.0%",
            "width": "11.3%",
            "height": "3.8%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "9": {
          "type": "WIDGET",
          "widgetId": "widget_card_tarifa",
          "isPending": false,
          "bbox": {
            "left": "74%",
            "right": "1.0%",
            "top": "87%",
            "bottom": "1.0%",
            "width": "25%",
            "height": "12%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "10": {
          "type": "WIDGET",
          "widgetId": "widget_btn_tarifas",
          "isPending": false,
          "bbox": {
            "left": "75%",
            "right": "13.7%",
            "top": "94.2%",
            "bottom": "2.0%",
            "width": "11.3%",
            "height": "3.8%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "11": {
          "type": "WIDGET",
          "widgetId": "widget_btn_lojas",
          "isPending": false,
          "bbox": {
            "left": "87%",
            "right": "1.7%",
            "top": "94.2%",
            "bottom": "2.0%",
            "width": "11.3%",
            "height": "3.8%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        }
      },
      "order": [
        "0",
        "1",
        "2",
        "3",
        "4",
        "5",
        "6",
        "7",
        "8",
        "9",
        "10",
        "11"
      ],
      "label": "Linhas e Horários"
    }
  },
  "widgets": {
    "widget_tit_lista": {
      "uri": "widgets/common/text/",
      "version": "1.19.0",
      "label": "Título – lista",
      "id": "widget_tit_lista",
      "config": {
        "text": "<p style=\"text-align:center;margin:0 0 4px 0\"><span style=\"font-size:20px;font-weight:700;color:#4463a9\">Linhas de Ônibus</span></p><p style=\"text-align:center;margin:0\"><span style=\"font-size:12px;color:#3d4657\">Clique numa linha para ver o trajeto e os horários</span></p>",
        "style": {
          "verticalAlign": "center",
          "wrap": true,
          "overflow": "auto",
          "padding": "p-1"
        }
      }
    },
    "widget_card_icone": {
      "uri": "widgets/common/image/",
      "version": "1.19.0",
      "label": "Cartão – ícone ônibus",
      "id": "widget_card_icone",
      "config": {
        "functionConfig": {
          "altText": "Ônibus",
          "toolTip": "",
          "linkParam": {},
          "scale": "Fit",
          "imageParam": {
            "url": "",
            "originalUrl": "",
            "fileName": "onibus.svg",
            "originalName": "onibus.svg",
            "imgSourceType": "BY_URL"
          },
          "imgSourceType": "",
          "imageFillMode": "FIT"
        },
        "styleConfig": {}
      },
      "useDataSources": [
        {
          "dataSourceId": "dataSource_1-linhas_onibus",
          "mainDataSourceId": "dataSource_1-linhas_onibus",
          "rootDataSourceId": "dataSource_1",
          "fields": []
        }
      ],
      "useDataSourcesEnabled": true
    },
    "widget_card_texto": {
      "uri": "widgets/common/text/",
      "version": "1.19.0",
      "label": "Cartão – linha (2 linhas)",
      "id": "widget_card_texto",
      "config": {
        "text": "<p style=\"margin:0;line-height:1.25\"><span style=\"font-size:14px;font-weight:700;color:#1b2a4a\"><exp data-uniqueid=\"9b03ccf0_7509_4abf_b425_129017539d9a\" data-dsid=\"dataSource_1-linhas_onibus\" data-expression=\"%7B%22name%22%3A%22%7BCOD_LINHA%7D%22%2C%22parts%22%3A%5B%7B%22type%22%3A%22FIELD%22%2C%22exp%22%3A%22%7BCOD_LINHA%7D%22%2C%22dataSourceId%22%3A%22dataSource_1-linhas_onibus%22%2C%22jimuFieldName%22%3A%22cod_linha%22%2C%22isFromRepeatedDataSourceContext%22%3Atrue%7D%5D%7D\">﻿<span contenteditable=\"false\">﻿﻿{COD_LINHA}﻿﻿</span>﻿</exp> - <exp data-uniqueid=\"5d716573_4012_4e9c_87b2_4b45da3a08a1\" data-dsid=\"dataSource_1-linhas_onibus\" data-expression=\"%7B%22name%22%3A%22%7BNOME%7D%22%2C%22parts%22%3A%5B%7B%22type%22%3A%22FIELD%22%2C%22exp%22%3A%22%7BNOME%7D%22%2C%22dataSourceId%22%3A%22dataSource_1-linhas_onibus%22%2C%22jimuFieldName%22%3A%22nome%22%2C%22isFromRepeatedDataSourceContext%22%3Atrue%7D%5D%7D\">﻿<span contenteditable=\"false\">﻿﻿{NOME}﻿﻿</span>﻿</exp></span></p><p style=\"margin:2px 0 0 0;line-height:1.25\"><span style=\"font-size:12px;color:#6b7690\">Sentido para <exp data-uniqueid=\"055d5ce2_17e7_41c3_ac6d_f6041de91d1d\" data-dsid=\"dataSource_1-linhas_onibus\" data-expression=\"%7B%22name%22%3A%22%7BSENTIDO%7D%22%2C%22parts%22%3A%5B%7B%22type%22%3A%22FIELD%22%2C%22exp%22%3A%22%7BSENTIDO%7D%22%2C%22dataSourceId%22%3A%22dataSource_1-linhas_onibus%22%2C%22jimuFieldName%22%3A%22sentido%22%2C%22isFromRepeatedDataSourceContext%22%3Atrue%7D%5D%7D\">﻿<span contenteditable=\"false\">﻿﻿{SENTIDO}﻿﻿</span>﻿</exp></span></p>",
        "style": {
          "verticalAlign": "center",
          "wrap": true,
          "overflow": "auto",
          "padding": "p-1"
        }
      },
      "useDataSources": [
        {
          "dataSourceId": "dataSource_1-linhas_onibus",
          "mainDataSourceId": "dataSource_1-linhas_onibus",
          "rootDataSourceId": "dataSource_1",
          "fields": [
            "cod_linha",
            "nome",
            "sentido"
          ]
        }
      ],
      "useDataSourcesEnabled": true
    },
    "widget_lista": {
      "uri": "widgets/common/list/",
      "version": "1.19.0",
      "id": "widget_lista",
      "label": "Lista de linhas",
      "config": {
        "maxItems": 200,
        "space": 4,
        "verticalSpace": 4,
        "horizontalSpace": 10,
        "itemsPerPage": 20,
        "isItemStyleConfirm": true,
        "isInitialed": true,
        "pageStyle": "SCROLL",
        "itemStyle": "STYLE5",
        "hoverType": "HOVER0",
        "selectedStyle": "STYLE0",
        "differentOddEven": false,
        "scrollBarOpen": true,
        "direction": "VERTICAL",
        "alignType": "CENTER",
        "scrollStep": 2,
        "cardConfigs": {
          "DEFAULT": {
            "cardSize": {
              "LARGE": {
                "width": 421,
                "height": 64
              }
            },
            "enable": true,
            "backgroundStyle": {
              "border": {
                "border": {
                  "type": "solid",
                  "color": "",
                  "width": "0px"
                }
              },
              "borderRadius": {
                "number": [
                  8,
                  8,
                  8,
                  8
                ],
                "unit": "px"
              }
            },
            "listLayout": "Custom"
          },
          "HOVER": {
            "cardSize": {
              "LARGE": {
                "width": 421,
                "height": 64
              }
            },
            "enable": true,
            "backgroundStyle": {
              "border": {
                "border": {
                  "type": "solid",
                  "color": "",
                  "width": "0px"
                }
              },
              "borderRadius": {
                "number": [
                  8,
                  8,
                  8,
                  8
                ],
                "unit": "px"
              },
              "background": {
                "color": "#eaf0fb",
                "fillType": "fill",
                "image": {
                  "url": ""
                }
              }
            },
            "listLayout": "Auto"
          },
          "SELECTED": {
            "cardSize": {
              "LARGE": {
                "width": 421,
                "height": 64
              }
            },
            "backgroundStyle": {
              "border": {
                "borderLeft": {
                  "type": "solid",
                  "color": "#012a84",
                  "width": "4px"
                },
                "borderBottom": {
                  "type": "solid",
                  "color": "",
                  "width": "2px"
                }
              },
              "borderRadius": {
                "number": [
                  8,
                  8,
                  8,
                  8
                ],
                "unit": "px"
              },
              "background": {
                "color": "rgba(1,42,132,0.10)",
                "fillType": "fill",
                "image": {
                  "url": ""
                }
              }
            },
            "selectionMode": "SINGLE",
            "listLayout": "Auto"
          }
        },
        "style": {
          "id": "default"
        },
        "isCheckEmptyTemplate": false,
        "filters": null,
        "isShowAutoRefresh": false,
        "layoutType": "ROW",
        "keepAspectRatio": false,
        "searchOpen": true,
        "sortOpen": false,
        "showRefresh": true,
        "showSelectedOnlyOpen": false,
        "showClearSelected": true,
        "showRecordCount": false,
        "filterOpen": false,
        "sqlExprObj": null,
        "sortFields": null,
        "searchFields": [
          "id_rota",
          "nome",
          "sentido"
        ],
        "searchHint": "Nº ou nome da linha",
        "hidePageTotal": false,
        "navigatorOpen": false,
        "lockItemRatio": false
      },
      "dataActions": {
        "setFilter": {
          "enabled": true
        },
        "relatedData": {
          "enabled": true
        }
      },
      "useDataSourcesEnabled": true,
      "layouts": {
        "DEFAULT": {
          "LARGE": "layout_card"
        }
      },
      "useDataSources": [
        {
          "dataSourceId": "dataSource_1-linhas_onibus",
          "mainDataSourceId": "dataSource_1-linhas_onibus",
          "rootDataSourceId": "dataSource_1",
          "fields": [
            "cod_linha",
            "nome",
            "sentido",
            "id_rota"
          ]
        }
      ],
      "outputDataSources": [],
      "style": {
        "background": {
          "color": "#ffffff",
          "fillType": "fill",
          "image": {
            "url": ""
          }
        },
        "borderRadius": {
          "number": [
            12,
            12,
            12,
            12
          ],
          "unit": "px"
        },
        "border": {
          "type": "solid",
          "color": "#dfe5ef",
          "width": "1px"
        }
      }
    },
    "widget_mapa": {
      "uri": "widgets/arcgis/arcgis-map/",
      "version": "1.19.0",
      "id": "widget_mapa",
      "label": "Mapa",
      "config": {
        "toolConfig": {
          "canZoom": true,
          "canHome": true,
          "canSearch": false,
          "canNavigation": true,
          "canLocate": true,
          "canCompass": true,
          "canFullScreen": false,
          "canScaleBar": true
        },
        "isUseCustomMapState": false,
        "initialMapDataSourceID": "dataSource_1",
        "initialMapState": null,
        "scaleRange": null,
        "disablePopUp": true
      },
      "layouts": {
        "MapFixedLayout": {
          "LARGE": "layout_mapa1"
        }
      },
      "useDataSources": [
        {
          "dataSourceId": "dataSource_1",
          "mainDataSourceId": "dataSource_1"
        }
      ]
    },
    "widget_horarios": {
      "uri": "widgets/horarios-linha/",
      "version": "1.19.0",
      "id": "widget_horarios",
      "label": "Horários da Linha",
      "config": {
        "dataSourceId": "dataSource_1-linhas_onibus",
        "linhasUrl": "https://geo.apucarana.pr.gov.br/server/rest/services/Hosted/Mobilidade_Linhas_Onibus/FeatureServer/0",
        "horariosUrl": "https://geo.apucarana.pr.gov.br/server/rest/services/Hosted/Mobilidade_Horarios_Onibus/FeatureServer/0",
        "tema": "azul",
        "mobilibusUrl": "https://editor.mobilibus.com/web/detalhes-linha/4abp9",
        "mobilibusLinhas": {
          "110": "fpt",
          "115": "2cjn",
          "120": "fqg",
          "125": "fqa",
          "130": "fqp",
          "135": "fq7",
          "140": "fq3",
          "145": "fq4",
          "150": "fpo",
          "155": "fr0",
          "160": "fqt",
          "165": "fqo",
          "170": "fr1",
          "175": "6ckh",
          "195": "fpg",
          "200": "fpm",
          "205": "fps",
          "210": "fqr",
          "215": "hrte",
          "220": "fpr",
          "230": "fq8",
          "235": "fqk",
          "240": "fph",
          "270": "9t55",
          "320": "fpu",
          "400": "fqn",
          "500": "fq6",
          "998": "fqd",
          "999": "fqb"
        },
        "paradasUrl": "https://geo.apucarana.pr.gov.br/server/rest/services/Hosted/Mobilidade_Paradas_Onibus/FeatureServer/0",
        "paradasLayerId": "paradas_onibus",
        "paradasDistanciaM": 30,
        "paradasLigadoAoAbrir": false,
        "paradasModo": "campo"
      },
      "useMapWidgetIds": [
        "widget_mapa"
      ]
    },
    "widget_tit_serv": {
      "uri": "widgets/common/text/",
      "version": "1.19.0",
      "label": "Título – serviços",
      "id": "widget_tit_serv",
      "config": {
        "text": "<p style=\"margin:0\"><span style=\"font-size:16px;font-weight:700;color:#4463a9\">Serviços ao passageiro</span></p>",
        "style": {
          "verticalAlign": "center",
          "wrap": true,
          "overflow": "auto",
          "padding": "p-1"
        }
      }
    },
    "widget_card_integra": {
      "uri": "widgets/common/text/",
      "version": "1.19.0",
      "label": "Cartão – Apucarana Integrada",
      "id": "widget_card_integra",
      "config": {
        "text": "<p style=\"margin:0 0 3px 0\"><span style=\"font-size:13px;font-weight:700;color:#1b2a4a\">🔄 Apucarana Integrada</span></p><p style=\"margin:0;line-height:1.3\"><span style=\"font-size:12px;color:#3d4657\">Pague <strong>uma passagem</strong> e troque de linha em qualquer ponto da cidade, sem passar pelo Terminal — com o <strong>Cartão Passe Fácil</strong> ou o app <strong>BiPay</strong>.</span></p>",
        "style": {
          "verticalAlign": "start",
          "wrap": true,
          "overflow": "auto",
          "padding": "p-2"
        }
      },
      "style": {
        "background": {
          "color": "#f4f6fb",
          "fillType": "fill",
          "image": {
            "url": ""
          }
        },
        "borderRadius": {
          "number": [
            12,
            12,
            12,
            12
          ],
          "unit": "px"
        },
        "border": {
          "type": "solid",
          "color": "#dfe5ef",
          "width": "1px"
        }
      }
    },
    "widget_card_app": {
      "uri": "widgets/common/text/",
      "version": "1.19.0",
      "label": "Cartão – app Vou de Ônibus",
      "id": "widget_card_app",
      "config": {
        "text": "<p style=\"margin:0 0 3px 0\"><span style=\"font-size:13px;font-weight:700;color:#1b2a4a\">📱 Vou de Ônibus Apucarana</span></p><p style=\"margin:0;line-height:1.3\"><span style=\"font-size:12px;color:#3d4657\">Horários, mapa da linha e previsão de chegada no celular.</span></p>",
        "style": {
          "verticalAlign": "start",
          "wrap": true,
          "overflow": "auto",
          "padding": "p-2"
        }
      },
      "style": {
        "background": {
          "color": "#f4f6fb",
          "fillType": "fill",
          "image": {
            "url": ""
          }
        },
        "borderRadius": {
          "number": [
            12,
            12,
            12,
            12
          ],
          "unit": "px"
        },
        "border": {
          "type": "solid",
          "color": "#dfe5ef",
          "width": "1px"
        }
      }
    },
    "widget_btn_gplay": {
      "uri": "widgets/common/button/",
      "version": "1.19.0",
      "id": "widget_btn_gplay",
      "label": "Google Play",
      "config": {
        "functionConfig": {
          "text": "Google Play",
          "toolTip": "Baixar o Vou de Ônibus Apucarana (Android)",
          "linkParam": {
            "linkType": "WEB_ADDRESS",
            "value": "https://play.google.com/store/apps/details?id=com.apucarana.voudeonibus",
            "openType": "_blank"
          }
        },
        "styleConfig": {
          "useCustom": true,
          "customStyle": {
            "regular": {
              "background": {
                "color": "#4463a9",
                "fillType": "fill"
              },
              "border": {
                "type": "solid",
                "color": "#4463a9",
                "width": "0px"
              },
              "text": {
                "color": "#ffffff",
                "size": "12px"
              },
              "borderRadius": {
                "number": [
                  100,
                  100,
                  100,
                  100
                ],
                "unit": "px"
              },
              "iconProps": {
                "color": "#fff",
                "size": 12
              }
            },
            "hover": {
              "background": {
                "color": "#012a84",
                "fillType": "fill"
              },
              "border": {
                "type": "solid",
                "color": "#012a84",
                "width": "0px"
              },
              "text": {
                "color": "#ffffff",
                "size": "12px"
              },
              "borderRadius": {
                "number": [
                  100,
                  100,
                  100,
                  100
                ],
                "unit": "px"
              },
              "iconProps": {
                "color": "#fff",
                "size": 12
              }
            }
          }
        }
      }
    },
    "widget_btn_appstore": {
      "uri": "widgets/common/button/",
      "version": "1.19.0",
      "id": "widget_btn_appstore",
      "label": "App Store",
      "config": {
        "functionConfig": {
          "text": "App Store",
          "toolTip": "Baixar o Vou de Ônibus Apucarana (iPhone)",
          "linkParam": {
            "linkType": "WEB_ADDRESS",
            "value": "https://apps.apple.com/br/app/vou-de-%C3%B4nibus-apucarana/id1488768260",
            "openType": "_blank"
          }
        },
        "styleConfig": {
          "useCustom": true,
          "customStyle": {
            "regular": {
              "background": {
                "color": "#4463a9",
                "fillType": "fill"
              },
              "border": {
                "type": "solid",
                "color": "#4463a9",
                "width": "0px"
              },
              "text": {
                "color": "#ffffff",
                "size": "12px"
              },
              "borderRadius": {
                "number": [
                  100,
                  100,
                  100,
                  100
                ],
                "unit": "px"
              },
              "iconProps": {
                "color": "#fff",
                "size": 12
              }
            },
            "hover": {
              "background": {
                "color": "#012a84",
                "fillType": "fill"
              },
              "border": {
                "type": "solid",
                "color": "#012a84",
                "width": "0px"
              },
              "text": {
                "color": "#ffffff",
                "size": "12px"
              },
              "borderRadius": {
                "number": [
                  100,
                  100,
                  100,
                  100
                ],
                "unit": "px"
              },
              "iconProps": {
                "color": "#fff",
                "size": 12
              }
            }
          }
        }
      }
    },
    "widget_card_tarifa": {
      "uri": "widgets/common/text/",
      "version": "1.19.0",
      "label": "Cartão – tarifas e lojas",
      "id": "widget_card_tarifa",
      "config": {
        "text": "<p style=\"margin:0 0 3px 0\"><span style=\"font-size:13px;font-weight:700;color:#1b2a4a\">💳 Tarifas, cartão e recarga</span></p><p style=\"margin:0;line-height:1.3\"><span style=\"font-size:12px;color:#3d4657\">1ª via do Passe Fácil grátis na VAL, ao lado do Terminal Urbano.</span></p>",
        "style": {
          "verticalAlign": "start",
          "wrap": true,
          "overflow": "auto",
          "padding": "p-2"
        }
      },
      "style": {
        "background": {
          "color": "#f4f6fb",
          "fillType": "fill",
          "image": {
            "url": ""
          }
        },
        "borderRadius": {
          "number": [
            12,
            12,
            12,
            12
          ],
          "unit": "px"
        },
        "border": {
          "type": "solid",
          "color": "#dfe5ef",
          "width": "1px"
        }
      }
    },
    "widget_btn_tarifas": {
      "uri": "widgets/common/button/",
      "version": "1.19.0",
      "id": "widget_btn_tarifas",
      "label": "Tarifas",
      "config": {
        "functionConfig": {
          "text": "Tarifas",
          "toolTip": "Tabela de tarifas da Viação Apucarana (VAL)",
          "linkParam": {
            "linkType": "WEB_ADDRESS",
            "value": "http://www.valap.com.br/tarifas.html",
            "openType": "_blank"
          }
        },
        "styleConfig": {
          "useCustom": true,
          "customStyle": {
            "regular": {
              "background": {
                "color": "#4463a9",
                "fillType": "fill"
              },
              "border": {
                "type": "solid",
                "color": "#4463a9",
                "width": "0px"
              },
              "text": {
                "color": "#ffffff",
                "size": "12px"
              },
              "borderRadius": {
                "number": [
                  100,
                  100,
                  100,
                  100
                ],
                "unit": "px"
              },
              "iconProps": {
                "color": "#fff",
                "size": 12
              }
            },
            "hover": {
              "background": {
                "color": "#012a84",
                "fillType": "fill"
              },
              "border": {
                "type": "solid",
                "color": "#012a84",
                "width": "0px"
              },
              "text": {
                "color": "#ffffff",
                "size": "12px"
              },
              "borderRadius": {
                "number": [
                  100,
                  100,
                  100,
                  100
                ],
                "unit": "px"
              },
              "iconProps": {
                "color": "#fff",
                "size": 12
              }
            }
          }
        }
      }
    },
    "widget_btn_lojas": {
      "uri": "widgets/common/button/",
      "version": "1.19.0",
      "id": "widget_btn_lojas",
      "label": "Lojas e recarga",
      "config": {
        "functionConfig": {
          "text": "Lojas e recarga",
          "toolTip": "Lojas e postos de recarga da VAL",
          "linkParam": {
            "linkType": "WEB_ADDRESS",
            "value": "http://www.valap.com.br/lojas/9/apucarana/48/centro.html",
            "openType": "_blank"
          }
        },
        "styleConfig": {
          "useCustom": true,
          "customStyle": {
            "regular": {
              "background": {
                "color": "#4463a9",
                "fillType": "fill"
              },
              "border": {
                "type": "solid",
                "color": "#4463a9",
                "width": "0px"
              },
              "text": {
                "color": "#ffffff",
                "size": "12px"
              },
              "borderRadius": {
                "number": [
                  100,
                  100,
                  100,
                  100
                ],
                "unit": "px"
              },
              "iconProps": {
                "color": "#fff",
                "size": 12
              }
            },
            "hover": {
              "background": {
                "color": "#012a84",
                "fillType": "fill"
              },
              "border": {
                "type": "solid",
                "color": "#012a84",
                "width": "0px"
              },
              "text": {
                "color": "#ffffff",
                "size": "12px"
              },
              "borderRadius": {
                "number": [
                  100,
                  100,
                  100,
                  100
                ],
                "unit": "px"
              },
              "iconProps": {
                "color": "#fff",
                "size": 12
              }
            }
          }
        }
      }
    }
  },
  "messageConfigs": {
    "messageConfig_lista_zoom": {
      "id": "messageConfig_lista_zoom",
      "widgetId": "widget_lista",
      "messageType": "DATA_RECORDS_SELECTION_CHANGE",
      "actions": [
        {
          "actionId": "widget_mapa-zoomToFeature-1",
          "widgetId": "widget_mapa",
          "messageWidgetId": "widget_lista",
          "actionName": "zoomToFeature",
          "description": null,
          "version": "1.19.0",
          "config": {
            "useDataSource": {
              "dataSourceId": "dataSource_1-linhas_onibus",
              "mainDataSourceId": "dataSource_1-linhas_onibus",
              "rootDataSourceId": "dataSource_1"
            },
            "useDataSources": [
              {
                "dataSourceId": "dataSource_1-linhas_onibus",
                "mainDataSourceId": "dataSource_1-linhas_onibus",
                "rootDataSourceId": "dataSource_1"
              }
            ],
            "goToInitialMapExtentWhenSelectionCleared": true,
            "useAnyTriggerData": true
          }
        },
        {
          "actionId": "widget_mapa-flash-2",
          "widgetId": "widget_mapa",
          "messageWidgetId": "widget_lista",
          "actionName": "flash",
          "description": null,
          "version": "1.19.0",
          "config": {
            "useAnyTriggerData": true,
            "messageUseDataSource": {
              "dataSourceId": "dataSource_1-linhas_onibus",
              "mainDataSourceId": "dataSource_1-linhas_onibus",
              "rootDataSourceId": "dataSource_1"
            },
            "actionUseDataSource": null,
            "sqlExprObj": null,
            "enabledDataRelationShip": true,
            "connectionType": "SET_CUSTOM_FIELDS"
          }
        }
      ]
    }
  },
  "dataSources": {
    "dataSource_1": {
      "id": "dataSource_1",
      "type": "WEB_MAP",
      "sourceLabel": "Mobilidade Urbana",
      "itemId": "c80cb05e18a645609f8f84740078b859",
      "portalUrl": "https://geo.apucarana.pr.gov.br/portal",
      "isHidden": false,
      "childDataSourceJsons": {
        "linhas_onibus": {
          "id": "dataSource_1-linhas_onibus",
          "type": "FEATURE_LAYER",
          "sourceLabel": "Linhas de ônibus",
          "portalUrl": "https://geo.apucarana.pr.gov.br/portal",
          "itemId": "8c163a4936e143be96f69b58a2911ce8",
          "layerId": "0",
          "url": "https://geo.apucarana.pr.gov.br/server/rest/services/Hosted/Mobilidade_Linhas_Onibus/FeatureServer/0"
        }
      }
    }
  }
}''')

with open(CFG_MODELO, encoding="utf-8") as f:
    cctv = json.load(f)
cfg, copiados, excluidos = montar_com_casca(cctv, BODY)
log("  Casca copiada do app 2:", ", ".join(copiados))
log("  Deixados de fora:", ", ".join(excluidos) or "-")
log("  Páginas:", " | ".join(p["label"] for p in cfg["pages"].values()))
log("  Widgets:", len(cfg["widgets"]))
log("✔ CÉLULA 3 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 4 — Backup e escrita do config do app 3 =====
log("\n▶ CÉLULA 4 — Escrita no app 3: iniciando...")
app3 = os.path.join(EXB_APPS, APP_NOVO)
for nome in ("config.json", os.path.join("resources", "config", "config.json"), "info.json"):
    o = os.path.join(app3, nome)
    if os.path.isfile(o):
        d = os.path.join(PASTA_BKP, "app3_" + nome.replace(os.sep, "_"))
        shutil.copy2(o, d); log("  Backup:", d)
img_o = os.path.join(EXB_APPS, APP_MODELO, "resources", "images")
if os.path.isdir(img_o):
    shutil.copytree(img_o, os.path.join(app3, "resources", "images"), dirs_exist_ok=True)
hc = cfg["widgets"]["widget_horarios"]["config"]
hc.update({"paradasUrl": PARADAS_URL, "paradasLayerId": "paradas_onibus", "paradasModo": "campo"})
log("  Widget Horários → paradas:", PARADAS_URL)
cfg["timestamp"] = int(datetime.datetime.now().timestamp() * 1000)
with open(CFG_NOVO, "w", encoding="utf-8") as f:
    json.dump(cfg, f, ensure_ascii=False, indent=2)
log("  config.json escrito:", CFG_NOVO)
log("✔ CÉLULA 4 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 5 — Web map: desligar pop-ups =====
log("\n▶ CÉLULA 5 — Pop-ups do web map: iniciando...")
from arcgis.gis import GIS
gis = GIS("pro")
WEBMAP_ID = "c80cb05e18a645609f8f84740078b859"
item = gis.content.get(WEBMAP_ID)
wm = item.get_data()
with open(os.path.join(PASTA_BKP, "webmap_antes.json"), "w", encoding="utf-8") as f:
    json.dump(wm, f, ensure_ascii=False, indent=2)
for ol in wm.get("operationalLayers", []):
    ol["disablePopup"] = True
    log("  · pop-up desligado:", ol.get("title"))
item.update(item_properties={"text": json.dumps(wm)})
log("✔ CÉLULA 5 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 6 — Próximos passos e links =====
log("\n▶ CÉLULA 6 — Próximos passos: iniciando...")
log("  1) REINICIE o client do ExB (widget novo): feche a janela do client e rode o Iniciar_ExB.bat de novo;")
log("     espere terminar a compilação (procure 'horarios-linha' sem erros na janela do client).")
log("  2) Construtor: Ctrl+F5 → mudança pequena → Salvar → Publicar")
log("  3) Teste: ligue Mostrar paradas e selecione a 195 sentido Terminal → devem aparecer 26 paradas")
log("\n  ===== LINKS =====")
log("   Construtor (app 3): https://localhost:3001/builder/?id=3")
log("   Prévia local:       https://localhost:3001/experience/3/?locale=pt-br")
log(f"   Item Paradas: https://geo.apucarana.pr.gov.br/portal/home/item.html?id={PARADAS_ID}")
log(f"\n  Backups em: {PASTA_BKP}")
log(f"  Saída completa: {ARQ_TXT}")
log("✔ CÉLULA 6 concluída com sucesso.")
